In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Lakehouse Implementation")
    .master("spark://spark-master:7077")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.hadoop:hadoop-aws:3.5.0",
            "software.amazon.awssdk:bundle:2.35.4",
            "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Make Iceberg the default table format for spark_catalog (Hive Metastore backed)
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog")
    .config("spark.sql.catalog.spark_catalog.type", "hive")
    .config("spark.sql.catalog.spark_catalog.uri", "thrift://hive-metastore:9083")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")      # Hive Metastore connection
    .enableHiveSupport()

    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "12345678")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.warehouse.dir", "s3a://buck1/warehouse")   # warehouse location

    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c700def-19de-43a3-ac36-3f11fb41ce04;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.

In [3]:
print(
    spark.sql(
        "SHOW DATABASES"
    ).show()
)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+

None


In [4]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS demo
""")

DataFrame[]

In [10]:
spark.sql("""
SHOW Tables in demo
""").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     demo|        customers|      false|
|     demo|           orders|      false|
|     demo|customers_iceberg|      false|
|     demo|   orders_iceberg|      false|
+---------+-----------------+-----------+



In [6]:
# Use the demo database
spark.sql("USE demo")

# ---------- Table 1: customers ----------
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.customers (
    customer_id INT,
    name STRING,
    city STRING
)
USING PARQUET
""")

spark.sql("""
INSERT INTO demo.customers VALUES
    (1, 'Ali Raza', 'Lahore'),
    (2, 'Sara Khan', 'Karachi'),
    (3, 'Bilal Ahmed', 'Islamabad'),
    (4, 'Mahnoor Iqbal', 'Lahore')
""")

# ---------- Table 2: orders ----------
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.orders (
    order_id INT,
    customer_id INT,
    amount DOUBLE,
    order_date STRING
)
USING PARQUET
""")

spark.sql("""
INSERT INTO demo.orders VALUES
    (101, 1, 250.50, '2026-01-05'),
    (102, 2, 89.99,  '2026-01-06'),
    (103, 1, 430.00, '2026-01-10'),
    (104, 3, 12.75,  '2026-02-01'),
    (105, 4, 999.00, '2026-02-03')
""")

# ---------- Query 1: simple select ----------
spark.sql("SELECT * FROM demo.customers").show()

# ---------- Query 2: join + aggregation ----------
spark.sql("""
SELECT c.name, c.city, COUNT(o.order_id) AS num_orders, SUM(o.amount) AS total_spent
FROM demo.customers c
JOIN demo.orders o ON c.customer_id = o.customer_id
GROUP BY c.name, c.city
ORDER BY total_spent DESC
""").show()

26/08/11 12:46:05 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+-----------+-------------+---------+
|customer_id|         name|     city|
+-----------+-------------+---------+
|          3|  Bilal Ahmed|Islamabad|
|          4|Mahnoor Iqbal|   Lahore|
|          2|    Sara Khan|  Karachi|
|          1|     Ali Raza|   Lahore|
+-----------+-------------+---------+



+-------------+---------+----------+-----------+
|         name|     city|num_orders|total_spent|
+-------------+---------+----------+-----------+
|Mahnoor Iqbal|   Lahore|         1|      999.0|
|     Ali Raza|   Lahore|         2|      680.5|
|    Sara Khan|  Karachi|         1|      89.99|
|  Bilal Ahmed|Islamabad|         1|      12.75|
+-------------+---------+----------+-----------+



In [7]:
print(spark.conf.get("spark.sql.catalogImplementation"))

hive


In [8]:
spark.sql("DESCRIBE FORMATTED demo.customers").show(100, truncate=False)

+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|customer_id                 |int                                                           |NULL   |
|name                        |string                                                        |NULL   |
|city                        |string                                                        |NULL   |
|                            |                                                              |       |
|# Detailed Table Information|                                                              |       |
|Catalog                     |spark_catalog                                                 |       |
|Database                    |demo                                                

In [ ]:
# currently tables are plain Parquet + Hive Metastore, no Iceberg yet.

In [6]:
spark.sql("USE demo")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.customers_iceberg (
    customer_id INT,
    name STRING,
    city STRING
)
USING ICEBERG
""")

spark.sql("""
INSERT INTO demo.customers_iceberg VALUES
    (1, 'Ali Raza', 'Lahore'),
    (2, 'Sara Khan', 'Karachi'),
    (3, 'Bilal Ahmed', 'Islamabad')
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.orders_iceberg (
    order_id INT,
    customer_id INT,
    amount DOUBLE,
    order_date STRING
)
USING ICEBERG
""")

spark.sql("""
INSERT INTO demo.orders_iceberg VALUES
    (101, 1, 250.50, '2026-01-05'),
    (102, 2, 89.99,  '2026-01-06'),
    (103, 1, 430.00, '2026-01-10')
""")

spark.sql("SELECT * FROM demo.customers_iceberg").show()
spark.sql("""
SELECT c.name, SUM(o.amount) AS total_spent
FROM demo.customers_iceberg c JOIN demo.orders_iceberg o ON c.customer_id = o.customer_id
GROUP BY c.name
""").show()

+-----------+-----------+---------+
|customer_id|       name|     city|
+-----------+-----------+---------+
|          1|   Ali Raza|   Lahore|
|          2|  Sara Khan|  Karachi|
|          3|Bilal Ahmed|Islamabad|
+-----------+-----------+---------+



+---------+-----------+
|     name|total_spent|
+---------+-----------+
| Ali Raza|      680.5|
|Sara Khan|      89.99|
+---------+-----------+



In [4]:
spark.sql("DESCRIBE FORMATTED demo.customers_iceberg").show(100, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|customer_id                 |int                                                                                                                   |NULL   |
|name                        |string                                                                                                                |NULL   |
|city                        |string                                                                                                                |NULL   |
|                            |                      

In [12]:
spark.sql("DROP TABLE IF EXISTS demo.customers")
spark.sql("DROP TABLE IF EXISTS demo.orders")

DataFrame[]

In [5]:
spark.sql("""
SHOW Tables in demo
""").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     demo|customers_iceberg|      false|
|     demo|   orders_iceberg|      false|
+---------+-----------------+-----------+



In [14]:
spark.conf.get("spark.sql.warehouse.dir")

's3a://buck1/warehouse'

## Kafka Implementation

In [5]:
kafka_df = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers",  "broker:9092") \
  .option("subscribe", "events") \
  .load()

In [6]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)
from pyspark.sql import functions as F


event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
])

In [7]:
events_df = (
    kafka_df
    .select(
        F.from_json(
            F.col("value").cast("string"),
            event_schema
        ).alias("data")
    )
    .select("data.*")
    .withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )
)

In [10]:
query = (
    events_df
    .writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation","s3a://buck1/checkpoints/events-v22")
    .trigger(processingTime="10 seconds")
    .toTable(
        "spark_catalog.demo.events"
    )
)
# query.awaitTermination()

In [19]:
spark.sql("""
SELECT * FROM demo.events
""").show()

+--------+-----------+----------+------+----------+
|event_id|customer_id|event_type|amount|event_time|
+--------+-----------+----------+------+----------+
+--------+-----------+----------+------+----------+



In [ ]:
spark.sql("DESCRIBE FORMATTED demo.customers_iceberg").show(100, truncate=False)

In [ ]:
# to store stream in kafka
kafka_df.writeStream \
    .format("kafka") \
    .option("checkpointLocation", "/app/checkpoint/") \
    .option("topic", "spark_stream_topic") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start() \
    .awaitTermination()

In [37]:
spark.sql("DROP TABLE IF EXISTS spark_catalog.demo.events")

DataFrame[]

In [38]:
spark.sql("SHOW TABLES IN spark_catalog.demo").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     demo|customers_iceberg|      false|
|     demo|   orders_iceberg|      false|
+---------+-----------------+-----------+



In [36]:
spark.sql("""
    DESCRIBE EXTENDED spark_catalog.demo.events
""").show(100, truncate=False)

+----------------------------+-------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                              |comment|
+----------------------------+-------------------------------------------------------------------------------------------------------+-------+
|event_id                    |string                                                                                                 |NULL   |
|customer_id                 |int                                                                                                    |NULL   |
|event_type                  |string                                                                                                 |NULL   |
|amount                      |double                                                                                                 |NULL   |